# Evaluacija ViT i Swin modela

In [ ]:
import time
import pandas as pd
from pathlib import Path
import json

from sklearn.metrics import precision_recall_fscore_support
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt



/home/vlada-maric/Desktop/mlProject/nnImgClas/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
BASE_DIR = Path("/home/vlada-maric/Desktop/mlProject/nnImgClas/03-evaluation")
TRAINED_MODELS_DIR = BASE_DIR / "trained_models"
DATA_LOCATION = BASE_DIR / "tiny-imagenet-200-modified" / "test_holdout"

NUM_CLASSES = 200
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [7]:
#Funkcija kojom ucitavamo podatke za testiranje
def load_dataset():
    normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #definisemo parametre za noramlizaciju podataka

    #defomosemo transformacije koje primenjujemo na svaku sliku
    test_transform = transforms.Compose([
        transforms.Resize(224), #resizujemo slike jer mreze ocekuju slike 224x224
        transforms.ToTensor(),
        normalization
    ])

    #ImageFolder ocekuje posbnu strukturu foldera koji sadrzi podatke
    validation_dataset = datasets.ImageFolder(DATA_LOCATION, transform = test_transform)

    #definismo interabilnu strkturu koja sadrzi podatke organizovane u batchove, za treniranje i za validaciju
    test_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=False)

    return test_loader

In [8]:
#Funkcija kojom ucitavamo model
def load_model(model_name):
    location = TRAINED_MODELS_DIR / model_name / "best_model.pth" #lokacija modela

    model_state = torch.load(location, map_location=DEVICE) #ucitavmo stanje modela

    #kreiramo modela i stavljamo model u eval stanje
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(model_state)
    model.to(DEVICE)
    model.eval()
    return model


In [ ]:

#Evaluacija modela
def evaluate_model(model,testset):
    lossFunction = nn.CrossEntropyLoss()

    total_loss = 0.0
    total = 0
    correct_top1 = 0
    top1_predicts = []
    true_labels = []

    #vreme za racunanje throughput_img_s
    start_time = time.time()

    #prolazimo kroz ceo testset
    with torch.no_grad():
      for images, labels in testset:
          images, labels = images.to(DEVICE), labels.to(DEVICE)

          model_predictons = model(images) #predvidjanja modela
          loss = lossFunction(model_predictons,labels) #greska modela
          total_loss += loss.item() * images.size(0) #akumuliramo ukupnu gresku

          predicted = model_predictons.argmax(dim=1) #predvidjanja modela po klasama
          correct_top1 += (predicted == labels).sum().item() #brojimo koliko tacnih predvidjanja imamo
          
          top1_predicts.extend(predicted.cpu().tolist()) #stavljamo u odgovarajuce liste
          true_labels.extend(labels.cpu().tolist()) 
          
          total += images.size(0)

    total_time = time.time() - start_time

    #racunamo statistike. racunamo prosecnu vrednost precison, recall-a i f1
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, top1_predicts, average="macro"
    )

    #pravimo dict od metrika
    test_metrics = {
        "top1_acc": correct_top1 / total,
        "loss": total_loss / total,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "throughput_img_s": total / total_time
    }

    return test_metrics




In [ ]:
#Ucitavamo vit model i racunamo metrike
testset = load_dataset()
model = load_model("vit_small_patch16_224")
metrics = evaluate_model(model,testset)

In [ ]:
#cuvamo metrike kao .json
with open("eval_vit.json", "w", encoding="utf-8") as fajl:
    json.dump(metrics, fajl, indent=4)

In [ ]:
#Ucitavamo SWin model i racunamo metrike
testset = load_dataset()
model = load_model("swin_tiny_patch4_window7_224")
metrics = evaluate_model(model,testset)

In [ ]:
#cuvamo metrike kao .json
with open("eval_swin.json", "w", encoding="utf-8") as fajl:
    json.dump(metrics, fajl, indent=4)